# SRL Adjacency Matrix Visualization

This notebook demonstrates how to load a sample from `english_srl.jsonl`, convert its span-level semantic role labeling (SRL) arguments into a token-level dependency relation matrix, and visualize it as an Adjacency Matrix.

It supports two representation modes:
- **`span` mode**: Fills the entire range of tokens in the argument span horizontally.
- **`dependency` mode**: Maps the span to its syntactic head token using POS tags, plotting only a single cell.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Patch

def find_head_index(span, pos_tags, words):
    """
    Finds the syntactic head index of an argument span using POS-tag rules.
    Maps span-level SRL arguments to single token indices in the dependency relation matrix.
    """
    start, end = span
    span_len = end - start
    if span_len <= 1:
        return start
    
    sub_pos = pos_tags[start:end]
    sub_words = words[start:end]
    
    # Rule 1: Split by prepositions, conjunctions, subordinators, and infinitival 'to'
    split_indices = [i for i, pos in enumerate(sub_pos) if pos in ('IN', 'TO', 'CC', 'WDT', 'WP', 'WRB')]
    if split_indices:
        limit = split_indices[0]
        if limit > 0:
            sub_pos = sub_pos[:limit]
            sub_words = sub_words[:limit]
            span_len = limit
        else:
            # Skip the leading preposition and recurse on the rest
            sub_pos = sub_pos[1:]
            sub_words = sub_words[1:]
            start += 1
            span_len -= 1
            return find_head_index((start, start + span_len), pos_tags, words)
            
    # Rule 2: Find the last noun or pronoun in the remaining span
    noun_indices = [i for i, pos in enumerate(sub_pos) if pos.startswith('NN') or pos == 'PRP']
    if noun_indices:
        return start + noun_indices[-1]
        
    # Rule 3: Find the last verb
    verb_indices = [i for i, pos in enumerate(sub_pos) if pos.startswith('VB')]
    if verb_indices:
        return start + verb_indices[-1]
        
    # Rule 4: Find the last adjective
    adj_indices = [i for i, pos in enumerate(sub_pos) if pos.startswith('JJ')]
    if adj_indices:
        return start + adj_indices[-1]
        
    return start + span_len - 1

def plot_srl_adjacency_matrix(words, pos_tags, verbs, mode="span", output_path=None):
    """
    Plots the SRL Adjacency Matrix with dark-mode theme.
    """
    N = len(words)
    
    # Color palette matching reference
    role_colors = {
        'ARG0': '#d9383a',      # Red
        'ARG1': '#1f8a84',      # Teal/Cyan
        'ARGM-GOL': '#cc9d70',   # Tan/Beige
        'ARGM-MOD': '#d97c2b',   # Orange
        'ARGM-TMP': '#4d6eb2',   # Slate Blue
        'ARG2': '#3b75af',      # Blue
        'ARG3': '#8c564b',      # Brown
        'ARG4': '#9467bd',      # Purple
        'ARGM-LOC': '#2ca02c',   # Green
        'ARGM-MNR': '#bcbd22',   # Yellowish Olive
        'ARGM-DIR': '#17becf',   # Light Cyan
        'ARGM-DIS': '#e377c2',   # Pink
    }
    default_role_color = '#7d69a3'
    bg_color = '#0f1115'
    grid_color = '#1b1e24'
    text_color = '#a0a8b6'
    
    cells = {}
    unique_roles_in_plot = set()
    
    for v in verbs:
        pred_idx = v['clean_index']
        roleset = v['roleset']
        if pred_idx is None or pred_idx >= N:
            continue
            
        cells[(pred_idx, pred_idx)] = {'type': 'predicate', 'label': roleset}
        
        for arg in v.get('arguments', []):
            label = arg['label']
            if label == 'rel':
                continue
            spans = arg.get('clean_spans', [])
            if not spans:
                continue
                
            unique_roles_in_plot.add(label)
            
            if mode == "span":
                for span in spans:
                    start, end = span
                    for col_idx in range(start, end):
                        if (pred_idx, col_idx) == (pred_idx, pred_idx):
                            continue
                        cells[(pred_idx, col_idx)] = {'type': 'argument', 'label': label}
            else:
                head_idx = find_head_index(spans[0], pos_tags, words)
                if head_idx >= N:
                    continue
                if (pred_idx, head_idx) != (pred_idx, pred_idx):
                    cells[(pred_idx, head_idx)] = {'type': 'argument', 'label': label}

    fig, ax = plt.subplots(figsize=(12, 12), facecolor=bg_color)
    ax.set_facecolor(bg_color)
    
    # Draw minor grid lines
    ax.set_xticks(np.arange(N) - 0.5, minor=True)
    ax.set_yticks(np.arange(N) - 0.5, minor=True)
    ax.grid(which='minor', color=grid_color, linestyle='-', linewidth=0.5)
    
    ax.set_xticks(np.arange(N))
    ax.set_yticks(np.arange(N))
    
    labels = [f"[{i}] {words[i]}" for i in range(N)]
    ax.set_xticklabels(labels, rotation=45, ha='right', color=text_color, fontsize=9)
    ax.set_yticklabels(labels, color=text_color, fontsize=9)
    
    ax.invert_yaxis()
    
    for spine in ax.spines.values():
        spine.set_visible(False)
        
    for (y, x), cell in cells.items():
        if cell['type'] == 'predicate':
            rect = patches.Rectangle((x - 0.45, y - 0.45), 0.9, 0.9, linewidth=1.2, edgecolor='#ffd700', facecolor='none', zorder=3)
            ax.add_patch(rect)
            ax.text(x, y, cell['label'], color='#ffd700', ha='center', va='center', fontweight='bold', fontsize=7, zorder=4)
        elif cell['type'] == 'argument':
            role = cell['label']
            color = role_colors.get(role, default_role_color)
            rect = patches.Rectangle((x - 0.45, y - 0.45), 0.9, 0.9, facecolor=color, edgecolor='none', zorder=3)
            ax.add_patch(rect)
            ax.text(x, y, role, color='#ffffff', ha='center', va='center', fontweight='bold', fontsize=6.5, zorder=4)
            
    ax.set_xlabel("Argument (token index)", color=text_color, labelpad=20, fontsize=11, fontweight='bold')
    ax.set_ylabel("Predicate (token index)", color=text_color, labelpad=20, fontsize=11, fontweight='bold')
    
    title_suffix = " (Span Mode)" if mode == "span" else " (Dependency Mode)"
    fig.suptitle("Adjacency Matrix - Semantic Role Labeling" + title_suffix, color='#ffffff', fontsize=16, fontweight='bold', y=0.96)
    ax.set_title(f"[ [ {' '.join(words)} ] ]", color=text_color, fontsize=10.5, pad=25)

    legend_elements = []
    sorted_roles = sorted(list(unique_roles_in_plot), key=lambda r: (0 if r.startswith('ARG') and r[3:].isdigit() else 1, r))
    for r in sorted_roles:
        color = role_colors.get(r, default_role_color)
        legend_elements.append(Patch(facecolor=color, edgecolor='none', label=r))
    legend_elements.append(patches.Rectangle((0, 0), 1, 1, linewidth=1.5, edgecolor='#ffd700', facecolor='none', label='Predicate node'))
    
    legend = ax.legend(handles=legend_elements, title="Semantic Roles", loc='upper left', bbox_to_anchor=(1.04, 1), facecolor=bg_color, edgecolor='none')
    plt.setp(legend.get_title(), color=text_color, fontweight='bold', fontsize=10)
    for text in legend.get_texts():
        text.set_color(text_color)
        text.set_fontsize(9)
        
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, facecolor=bg_color, bbox_extra_artists=(legend,), bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# Load a sample from english_srl.jsonl and plot its SRL Adjacency Matrix
import json

jsonl_path = "../../english_srl.jsonl"
sample_index = 0  # Sentence 0 containing the long span

sample = None
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i == sample_index:
            sample = json.loads(line)
            break

if sample:
    words = sample['words']
    pos_tags = sample['pos_tags']
    verbs = sample['verbs']
    
    print(f"Sentence: {' '.join(words)}")
    
    print("\n--- 1. Span-based Visualization (Filled horizontal bars) ---")
    plot_srl_adjacency_matrix(words, pos_tags, verbs, mode="span")
    
    print("\n--- 2. Dependency-based Visualization (Single head token block) ---")
    plot_srl_adjacency_matrix(words, pos_tags, verbs, mode="dependency")
else:
    print("Could not find sample index in jsonl file.")